# **Mini Demo Walkthrough: Order Status Resolution Agent**

### **Overview**
This demonstration showcases the **end-to-end workflow of an intelligent support automation agent** designed to handle customer inquiries about order status.  
The scenario aligns directly with the **“Mini Demo Walkthrough” slide**, illustrating how each component of the agent maps to real-world business operations in customer service systems.

The demo emphasizes how an **Agentic AI system** integrates reasoning, data extraction, backend lookups, and CRM updates within a controlled, auditable workflow.  
It bridges the conceptual pipeline shown in the slide with executable logic that can be extended into production-ready automation.

---

### **Scenario**
**Business Context:**  
A customer support agent receives an email asking for the status of an order. The AI system must:
1. Identify the order ID automatically from the email.  
2. Retrieve the real-time status from a database or API.  
3. Draft a professional response to the customer.  
4. Log the interaction for future audit and reporting.

**Goal:** Automate repetitive support queries while maintaining transparency, accuracy, and compliance with responsible AI practices.

---

### **Demo Objectives**
- Simulate a **realistic customer interaction** using structured email input.  
- Demonstrate **data extraction and validation** (identifying order IDs from text).  
- Query a **mock order database** to retrieve relevant status information.  
- Generate a **personalized response** (template-based or LLM-assisted).  
- Record the full transaction in a **CRM log** for traceability.

---

### **Pipeline Alignment with Slide**

| Slide Step | Code Implementation | Description |
|-------------|--------------------|--------------|
| **Email Reception** | `Email` dataclass and `sample_email` | Represents the customer inquiry arriving in the support inbox. |
| **Data Extraction** | `extract_order_id()` | Automatically identifies and validates order IDs like `ORD-1045` using pattern recognition. |
| **Database Query** | `get_order_status()` | Simulates retrieving order details (status, tracking, ETA) from a backend system. |
| **Customer Update** | `compose_update_email()` | Creates a personalized update message; optionally uses an LLM for human-like phrasing. |
| **CRM Logging** | `log_interaction()` | Documents the entire transaction, ensuring accountability and future reference. |

---

### **Technical Explanation**

This demo is structured around modular, auditable functions — each representing a component of an enterprise-grade AI agent pipeline:

- **Email Object:** Captures sender, subject, and body with timestamps, representing input from real customer systems.  
- **Regex-Based Extraction:** Efficiently identifies order IDs without relying on external APIs or ML models.  
- **Mock Database Query:** Demonstrates system integration where an agent queries structured records.  
- **LLM-Assisted Communication:** Uses a language model (optional) to compose clear, customer-facing responses aligned with company tone.  
- **CRM Logging:** Stores every interaction in a structured CSV format, reinforcing transparency and traceability.

The structure models how **enterprise AI systems orchestrate deterministic steps with optional LLM reasoning** — ensuring that business logic and communication remain interpretable and auditable.

---

### **Ethical and Governance Alignment**
This demo not only illustrates workflow automation but also touches on **Responsible AI practices**:
- No sensitive data leaves the system without explicit handling.  
- Every action (data access, response generation, logging) is transparent.  
- The pipeline ensures accountability, allowing compliance with governance frameworks such as **ISO 42001, GDPR, or DPDP**.

---

### **Learning Outcome**
By the end of this demo, learners will:
- Understand how to structure a multi-step agent pipeline using modular, testable components.  
- Recognize how each pipeline step corresponds to a real-world process in automated customer service.  
- See how **Agentic AI systems combine symbolic logic, data integration, and natural language generation** within safe governance boundaries.

---




In [ ]:
import os, re, csv, datetime
from dataclasses import dataclass, asdict
from pathlib import Path

# --- Optional LLM phrasing (can be removed to keep it fully offline) ---
import os
os.environ["OPENAI_API_KEY"] = ""
os.environ["OPENAI_API_BASE"] = "https://openai.vocareum.com/v1"
USE_LLM = False
if USE_LLM:
    from langchain_openai import ChatOpenAI
    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

In [ ]:
# --------------------------
# 1) Email Reception (input)
# --------------------------
@dataclass
class Email:
    from_addr: str
    subject: str
    body: str
    received_at: datetime.datetime

sample_email = Email(
    from_addr="jane.doe@example.com",
    subject="Order status request: ORD-1045",
    body=(
        "Hello Support,\n"
        "I placed an order last week but haven’t received an update yet.\n"
        "Order ID: ORD-1045. Please share latest status and tracking link.\n"
        "Thanks,\nJane"
    ),
    received_at=datetime.datetime.now()
)

In [ ]:
# ---------------------------------------------------
# 2) Data Extraction: Order ID detection + validation
# ---------------------------------------------------
ORDER_ID_RE = re.compile(r"\bORD-\d{4,6}\b", flags=re.IGNORECASE)

def extract_order_id(text: str) -> str | None:
    m = ORDER_ID_RE.search(text)
    return m.group(0).upper() if m else None

In [ ]:
# -------------------------------
# 3) Database Query: mock backend
# -------------------------------
MOCK_DB = {
    "ORD-1045": {
        "status": "Shipped",
        "carrier": "Delhivery",
        "tracking_id": "DLV123456789IN",
        "eta": "2025-11-14",
        "items": [{"sku": "TSHIRT-RED-M", "qty": 1}]
    },
    "ORD-2048": {
        "status": "Processing",
        "carrier": None,
        "tracking_id": None,
        "eta": "2025-11-16",
        "items": [{"sku": "JNS-INDIGO-32", "qty": 1}, {"sku": "BELT-BLK-34", "qty": 1}]
    }
}

def get_order_status(order_id: str) -> dict | None:
    return MOCK_DB.get(order_id)

In [ ]:
# ------------------------------------------------------------
# 4) Customer Update: compose a personalized status message
# ------------------------------------------------------------
def compose_update_email(customer_email: str, order_id: str, rec: dict) -> str:
    if USE_LLM:
        prompt = (
            f"Write a concise, polite update email to {customer_email} about order {order_id}. "
            f"Status: {rec['status']}. Carrier: {rec.get('carrier')}. Tracking: {rec.get('tracking_id')}. "
            f"ETA: {rec.get('eta')}. Keep it under 120 words."
        )
        return llm.invoke(prompt).content
    # Deterministic template (no LLM)
    lines = [
        f"Subject: Update on your order {order_id}",
        "",
        f"Hi,",
        f"We’ve checked your order {order_id}. Current status: {rec['status']}.",
    ]
    if rec.get("carrier") and rec.get("tracking_id"):
        lines.append(f"Carrier: {rec['carrier']}, Tracking ID: {rec['tracking_id']}.")
        lines.append("Tracking link: https://tracking.example.com/" + rec["tracking_id"])
    if rec.get("eta"):
        lines.append(f"Estimated delivery date: {rec['eta']}.")
    lines.append("")
    lines.append("Thanks,\nSupport Team")
    return "\n".join(lines)

In [ ]:
# -----------------------------------
# 5) CRM Logging: CSV-based audit log
# -----------------------------------
LOG_PATH = Path("crm_log.csv")

def log_interaction(email: Email, order_id: str | None, outcome: str, notes: str):
    new = not LOG_PATH.exists()
    with LOG_PATH.open("a", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(
            f,
            fieldnames=["timestamp", "from_addr", "subject", "order_id", "outcome", "notes"]
        )
        if new:
            writer.writeheader()
        writer.writerow({
            "timestamp": datetime.datetime.now().isoformat(timespec="seconds"),
            "from_addr": email.from_addr,
            "subject": email.subject,
            "order_id": order_id or "",
            "outcome": outcome,
            "notes": notes[:500]
        })

In [ ]:
# ----------------------
# Orchestration (Agent)
# ----------------------
@dataclass
class AgentResult:
    order_id: str | None
    db_record: dict | None
    reply: str
    outcome: str

def handle_email(email: Email) -> AgentResult:
    # Extract
    oid = extract_order_id(email.subject + "\n" + email.body)

    if not oid:
        msg = ("Subject: Could you share your Order ID?\n\n"
               "Hi, we couldn’t locate an Order ID in your message. "
               "Please reply with your ID in the form ORD-XXXX.")
        log_interaction(email, None, "MISSING_ORDER_ID", "Requested customer for Order ID")
        return AgentResult(order_id=None, db_record=None, reply=msg, outcome="MISSING_ORDER_ID")

    # Query
    rec = get_order_status(oid)
    if not rec:
        msg = (f"Subject: Order {oid} not found\n\n"
               f"Hi, we couldn’t find {oid} in our system. Please confirm the ID or share the invoice.")
        log_interaction(email, oid, "ORDER_NOT_FOUND", "Asked for confirmation")
        return AgentResult(order_id=oid, db_record=None, reply=msg, outcome="ORDER_NOT_FOUND")

    # Compose update
    reply = compose_update_email(email.from_addr, oid, rec)
    log_interaction(email, oid, "ORDER_STATUS_SENT", f"Status={rec['status']}; ETA={rec.get('eta')}")
    return AgentResult(order_id=oid, db_record=rec, reply=reply, outcome="ORDER_STATUS_SENT")

In [ ]:
# -----------------------
# Run the slide-aligned flow
# -----------------------
result = handle_email(sample_email)

print("=== PIPELINE OUTPUT (Slide Steps) ===")
print("1) Email Reception: OK")
print(f"2) Data Extraction: order_id = {result.order_id}")
print(f"3) Database Query: found = {bool(result.db_record)}")
print(f"4) Customer Update (preview):\n{result.reply}\n")
print(f"5) CRM Logging: written to {LOG_PATH.resolve()}")
print(f"Outcome: {result.outcome}")

### **What Happens Behind the Scenes**

1. **Email Reception**  
   The agent begins by receiving a simulated customer inquiry (an email object).  
   This represents a message arriving in a real-world customer support inbox.

2. **Data Extraction**  
   The system automatically scans the email subject and body to detect an **Order ID** (e.g., `ORD-1045`).  
   A regular expression identifies valid order formats and ensures they are captured precisely before proceeding.

3. **Database Query**  
   Once an Order ID is found, the agent queries a **mock order database** to fetch order details — including status, carrier, tracking ID, and estimated delivery date.  
   This step simulates a secure backend lookup, typical of enterprise integrations with ERP or order management systems.

4. **Customer Update Generation**  
   The agent then prepares a **personalized response** for the customer:  
   - If an LLM is enabled, it crafts a natural, conversational email with the retrieved details.  
   - If not, it uses a deterministic message template for consistency and clarity.

5. **CRM Logging**  
   Every interaction (email address, order ID, outcome, timestamp) is **logged automatically** into a structured CSV file.  
   This ensures traceability, allowing customer service managers to audit the full interaction history.

6. **Pipeline Outcome**  
   The notebook prints a summary of each stage — confirming extraction, query success, response creation, and CRM entry.  
   These logs provide visibility into the **entire reasoning and data flow**, mirroring how real AI agents operate in production environments.

---

**In essence:**  
The agent reads an incoming request → extracts intent and identifiers → fetches relevant data → crafts a context-aware reply → and records everything for governance.  
The printed output in the notebook reflects this transparent, step-by-step workflow trace.
